# CFTC Nasdaq Mini 持仓映射的时钟修复

冻结源计划保留原映射；排除 2025-12-23 当前版本与依赖它的四周变化，建立新 Qlib Recorder。

In [1]:
import os,csv,io,json,hashlib,zipfile,pickle,tarfile,tempfile,subprocess,shlex
from datetime import datetime,timedelta
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.data.nas_market_data import _settings,archive_nas_recorder
root=Path.cwd();card=root/'research/stage1_cftc_nq_tff_14d_timing_repair_20261023';plan=json.loads((card/'plan.json').read_text());raw=root/'data/snapshots/stage1_cftc_nq_tff_source_probe_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and len(plan['raw_zip_sha256'])==16 and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
state={'status':'started','plan_sha256':sha(card/'plan.json'),'model_fits':0,'native_accounts':0,'market_outcome_reads':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
rows=[];headers={}
for year in range(2011,2027):
 p=raw/f'fut_fin_txt_{year}.zip';assert sha(p)==plan['raw_zip_sha256'][str(year)]
 with zipfile.ZipFile(p) as z:
  m=[x for x in z.namelist() if x.lower().endswith(('.txt','.csv'))];assert len(m)==1;body=z.read(m[0])
 try:s=body.decode('utf-8-sig')
 except UnicodeDecodeError:s=body.decode('cp1252')
 table=csv.DictReader(io.StringIO(s));cols=table.fieldnames;alias=[x for x in ('Report_Date_as_MM_DD_YYYY','Report_Date_as_YYYY-MM-DD') if x in cols];assert len(alias)==1;dates=alias[0]
 needed=['Market_and_Exchange_Names','CFTC_Contract_Market_Code','Open_Interest_All','Lev_Money_Positions_Long_All','Lev_Money_Positions_Short_All'];assert all(x in cols for x in needed)
 select=[x for x in table if x['CFTC_Contract_Market_Code'].strip()==plan['contract_code'] and all(k in x['Market_and_Exchange_Names'].upper() for k in ('NASDAQ','MINI','CHICAGO MERCANTILE'))];assert len(select)>=20
 num=lambda x:float(x.replace(',','').strip())
 for x in select:rows.append({'report_date':pd.Timestamp(x[dates].strip()),'open_interest':num(x['Open_Interest_All']),'leveraged_long':num(x['Lev_Money_Positions_Long_All']),'leveraged_short':num(x['Lev_Money_Positions_Short_All'])})
 headers[str(year)]={'alias':dates,'rows':len(select)}
weekly=pd.DataFrame(rows).sort_values('report_date').reset_index(drop=True);assert len(weekly)==824 and weekly.report_date.is_unique
weekly['net_share']=(weekly.leveraged_long-weekly.leveraged_short)/weekly.open_interest
release={pd.Timestamp(k):pd.Timestamp(v) for k,v in plan['availability']['2025_backlog_final_official_release_dates'].items()};assert len(release)==13
excluded=pd.Series(False,index=weekly.index)
for start,end in plan['availability']['unknown_report_date_ranges_inclusive']:excluded|=weekly.report_date.between(pd.Timestamp(start),pd.Timestamp(end))
excluded|=weekly.report_date.isin(pd.to_datetime(plan['availability']['additional_excluded_report_dates']))
weekly['date_excluded_unknown_clock']=excluded
weekly['release_date_proxy']=weekly.report_date+pd.Timedelta(days=14)
for d,r in release.items():
 mask=weekly.report_date.eq(d);assert mask.sum()==1;weekly.loc[mask,'release_date_proxy']=r+pd.Timedelta(days=1)
weekly.loc[excluded,'release_date_proxy']=pd.NaT
weekly['delta4']=weekly.net_share-weekly.net_share.shift(4)
valid=[];ready=[]
for i in range(len(weekly)):
 prior=weekly.iloc[max(0,i-4):i+1]
 ok=i>=4 and len(prior)==5 and prior.release_date_proxy.notna().all() and prior.delta4.notna().iloc[-1]
 valid.append(bool(ok));ready.append(prior.release_date_proxy.max() if ok else pd.NaT)
weekly['delta4_source_known']=valid;weekly['coordinate_available_date']=pd.to_datetime(ready)
assert weekly.loc[~weekly.delta4_source_known,'coordinate_available_date'].isna().all()
assert weekly.loc[weekly.delta4_source_known,'coordinate_available_date'].ge(weekly.loc[weekly.delta4_source_known,'report_date']).all()
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['Qlib']['provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-07',freq='day'));assert calendar.is_monotonic_increasing and calendar.is_unique
signal_clock=pd.DatetimeIndex(calendar).tz_localize('America/New_York')+pd.Timedelta(hours=9,minutes=45)
source=weekly.loc[weekly.delta4_source_known,['report_date','coordinate_available_date','delta4']].copy()
source['available_at_utc']=source.coordinate_available_date.dt.tz_localize('America/New_York').dt.tz_convert('UTC')
source=source.sort_values(['available_at_utc','report_date'])
assert source.available_at_utc.is_monotonic_increasing
left=pd.DataFrame({'signal_date':calendar,'signal_clock_utc':signal_clock.tz_convert('UTC')})
joined=pd.merge_asof(left,source[['available_at_utc','report_date','delta4']],left_on='signal_clock_utc',right_on='available_at_utc',direction='backward',allow_exact_matches=True)
joined.index=calendar;joined['source_age_days']=(joined.signal_date-joined.report_date).dt.days
joined['source_known']=joined.delta4.notna() & joined.source_age_days.le(plan['availability']['max_feature_age_calendar_days']) & joined.available_at_utc.le(joined.signal_clock_utc)
joined['CFTC_NQ_LevNetOI_delta4']=joined.delta4.where(joined.source_known)
assert joined.loc[joined.source_known,'CFTC_NQ_LevNetOI_delta4'].notna().all()
assert not joined.loc[~joined.source_known,'CFTC_NQ_LevNetOI_delta4'].notna().any()
train=joined.loc['2011-01-03':'2022-12-30'];held=joined.loc['2023-01-03':'2026-10-07'];assert len(train)==3020 and len(held)==944 and len(joined)==3964
static=joined[['CFTC_NQ_LevNetOI_delta4','source_known']].copy();static.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument']);static=pd.concat({'feature':static[['CFTC_NQ_LevNetOI_delta4']],'meta':static[['source_known']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2011-01-03','2022-12-30'),'held':('2023-01-03','2026-10-07')});assert len(dataset.prepare('train',col_set=['feature','meta']))==3020 and len(dataset.prepare('held',col_set=['feature','meta']))==944;dataset.config(dump_all=True,recursive=True)
report={'status':'development_only_conservative_CFTC_NQ_delta4_calendar_mapped_first_public_vintage_pending','plan_sha256':sha(card/'plan.json'),'weekly_report_rows':len(weekly),'source_coordinate_valid_rows':int(weekly.delta4_source_known.sum()),'unknown_clock_report_rows':int(excluded.sum()),'train_calendar_rows':len(train),'held_calendar_rows':len(held),'known_train_sessions':int(train.source_known.sum()),'known_held_sessions':int(held.source_known.sum()),'missing_train_sessions':int((~train.source_known).sum()),'missing_held_sessions':int((~held.source_known).sum()),'coordinate_train_min':float(train.CFTC_NQ_LevNetOI_delta4.min()),'coordinate_train_max':float(train.CFTC_NQ_LevNetOI_delta4.max()),'first_public_vintage_qualified':False,'historical_release_calendar_qualified':False,'model_fits':0,'native_accounts':0,'market_outcome_reads':0}
assert weekly.loc[weekly.report_date.eq(pd.Timestamp('2025-12-23')),'date_excluded_unknown_clock'].all()
assert joined.loc[pd.Timestamp('2025-12-30'),'report_date']<pd.Timestamp('2025-12-23')
with R.uri_context(str(root/'research/stage1_cftc_nq_tff_14d_positioning_map_20261023/qlib_recorder')):
 prior=R.get_recorder(recorder_id='cdd3e57a851a48d8bc563bac302084f5',experiment_id='108116900852740075').load_object('analysis/mapped_signals')
changed=~(joined['CFTC_NQ_LevNetOI_delta4'].eq(prior['CFTC_NQ_LevNetOI_delta4']) | (joined['CFTC_NQ_LevNetOI_delta4'].isna() & prior['CFTC_NQ_LevNetOI_delta4'].isna()))
report['changed_signal_rows_vs_parent_map']=int(changed.sum());report['changed_signal_first']=str(joined.index[changed][0]);report['changed_signal_last']=str(joined.index[changed][-1]);assert report['changed_signal_rows_vs_parent_map']>0
(card/'source_audit.json').write_text(json.dumps(report,indent=2)+'\n')
tracking=card/'qlib_recorder'
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,raw_hashes=plan['raw_zip_sha256']);rec.save_objects(artifact_path='analysis',weekly=weekly,mapped_signals=joined,source_audit=report);rec.save_objects(artifact_path='metadata',dataset=dataset);rec.log_metrics(source_train_days=len(train),source_held_days=len(held),known_train_days=int(train.source_known.sum()),known_held_days=int(held.source_known.sum()),model_fits=0,native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 rec=R.get_recorder(recorder_id=rid,experiment_id=eid);assert len(rec.load_object('analysis/mapped_signals'))==3964 and len(rec.load_object('metadata/dataset').prepare('held',col_set=['feature','meta']))==944
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='cftc-map-restore-') as td:
 archive=Path(td)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=120)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as z:names=[m.name for m in z.getmembers() if m.isfile()]
 assert any(x.endswith('/artifacts/analysis/mapped_signals') for x in names)
summary={'experiment':plan['experiment'],'decision':report['status'],'objective_achieved':False,'plan':'plan.json','source_audit':'source_audit.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state.update(status='completed',recorder_id=rid,nas_archive_sha256=evidence['sha256']);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(report);print({'recorder_id':rid,'nas_sha256':evidence['sha256']})


{'status': 'development_only_conservative_CFTC_NQ_delta4_calendar_mapped_first_public_vintage_pending', 'plan_sha256': 'e2bd03b4558042f962640fb40c8eb2877908e91f030425fb592aeec95737044c', 'weekly_report_rows': 824, 'source_coordinate_valid_rows': 790, 'unknown_clock_report_rows': 18, 'train_calendar_rows': 3020, 'held_calendar_rows': 944, 'known_train_sessions': 2947, 'known_held_sessions': 915, 'missing_train_sessions': 73, 'missing_held_sessions': 29, 'coordinate_train_min': -0.33269111915727184, 'coordinate_train_max': 0.30047364803745835, 'first_public_vintage_qualified': False, 'historical_release_calendar_qualified': False, 'model_fits': 0, 'native_accounts': 0, 'market_outcome_reads': 0, 'changed_signal_rows_vs_parent_map': 28, 'changed_signal_first': '2025-12-30 00:00:00', 'changed_signal_last': '2026-02-09 00:00:00'}
{'recorder_id': 'f11156af93d24fdbb942c0c37c5408ad', 'nas_sha256': '382b8b0a16f07a742f0a2384f9a2a281f77f39639c53147a21d0fd4a096f87dc'}


## 中文结论

时点修复后 28 个 Stage1 日期的持仓变化值改变，2025-12-30 不再使用 12-23 当前版本。完整日历、原生 Recorder 和 NAS 恢复通过；源首发时点仍未全面合格，零拟合、零账户。详见 `conclusion.md`。